# สมุดฝึกปฏิบัติการตัดเสียงสะท้อนในห้องคงที่ (Scenario S1: Static-Room Echo Cancellation)

> **โหมดการเรียนรู้ (CDX-04 & CDX-05)**: สมุดทดลองนี้จัดทำขึ้นเพื่อให้นักศึกษาทั้ง 2 คนกดรันทีละขั้น ทดลองเปลี่ยนค่าพารามิเตอร์ที่ปลอดภัย และอธิบายผลลัพธ์ด้วยตนเอง ควบคู่กับเอกสาร [`LEARNING_GUIDE_S1_TH.md`](../LEARNING_GUIDE_S1_TH.md)

### การแบ่งหน้าที่ของนักศึกษาทั้ง 2 คน
- **นักศึกษาคนที่ 1 (Student A)**: รับผิดชอบ Cell ที่ 1 และ Cell ที่ 2 (การโหลดเสียงต้นทาง `x[n]`, การสร้างผลตอบสนองห้องคงที่ `h_room[n]`, การจำลองเสียงไมโครโฟน `d[n] = x[n] * h_room[n]`, และการตรวจสอบกราฟ Waveform / Spectrogram)
- **นักศึกษาคนที่ 2 (Student B)**: รับผิดชอบ Cell ที่ 3 และ Cell ที่ 4 (การรันตัวกรองปรับตัวได้ LMS vs NLMS ภายใต้พารามิเตอร์เดียวกัน, การวิเคราะห์ผลของการหารด้วยพลังงานใน NLMS, และการคำนวณค่า ERLE / RMSE)
- **ทำร่วมกัน (Student A & B)**: รับผิดชอบ Cell ที่ 5 (การรันครบทั้ง 10 คลิป สรุปตารางผลลัพธ์ และซ้อมตอบคำถามปากเปล่าท้ายสมุด)

In [ ]:
# =============================================================================
# CELL 1 [Student A]: ตั้งค่าเริ่มต้น โหลดเสียงพูด 1 คลิป และสร้างห้องจำลองคงที่ h_room[n]
# =============================================================================
# สิ่งที่ควรเกิดขึ้นเมื่อรัน Cell นี้:
#   1. โหลดไฟล์เสียงพูดสะอาด 16 kHz Mono ความยาว 8.0 วินาที (128,000 จุด)
#   2. สร้างผลตอบสนองอิมพัลส์ของห้องคงที่ rir_0 ขนาด 128 taps (8.0 ms)
# ค่าที่นักศึกษาสามารถลองเปลี่ยนได้อย่างปลอดภัย:
#   - ลองเปลี่ยน SELECTED_CLIP_INDEX เป็นเลข 0 ถึง 9 เพื่อเลือกฟัง/ดูคลิปเสียงพูดอื่น

import sys
import wave
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.lms import create_fixed_room_ir, simulate_microphone_signal, run_lms, compute_lms_stability_bound
from src.nlms import run_nlms
from src.metrics import compute_erle_db, compute_rmse
from src.run_s1_experiment import prepare_10_clean_speech_clips

SAMPLE_RATE = 16000
SELECTED_CLIP_INDEX = 0  # <--- [Student A ลองเปลี่ยนเป็น 0..9 ได้]

clips_meta = prepare_10_clean_speech_clips()
clip_path = PROJECT_ROOT / str(clips_meta[SELECTED_CLIP_INDEX]["local_path"])

with wave.open(str(clip_path), "rb") as wf:
    x = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float64) / 32768.0

h_room = create_fixed_room_ir(length=128, rir_id="rir_0")
print(f"โหลดคลิปที่ {SELECTED_CLIP_INDEX}: {clip_path.name}, ความยาว = {len(x)} จุด ({len(x)/SAMPLE_RATE:.1f} s)")
print(f"สร้างห้องจำลองคงที่ rir_0: จำนวนสัมประสิทธิ์ = {len(h_room)} taps ({len(h_room)/SAMPLE_RATE*1000:.1f} ms)")

In [ ]:
# =============================================================================
# CELL 2 [Student A]: จำลองเสียงก้องที่ไมโครโฟน d[n] = x[n] * h_room[n]
# =============================================================================
# สิ่งที่ควรเกิดขึ้นเมื่อรัน Cell นี้:
#   - คำนวณสัญญาณไมโครโฟน d[n] จากการทำ Linear Convolution ระหว่าง x[n] กับ h_room[n]
#   - เนื่องจากใน S1 ไม่มีคนพูดฝั่งใกล้ (target = 0) ดังนั้นเมื่อตัดเสียงก้องออกแล้ว e[n] ที่ถูกต้องต้องเข้าใกล้ความเงียบ

d, echo, target = simulate_microphone_signal(x, [h_room], rir_switch_samples=None, nearend=None)

print(f"RMS ของเสียงลำโพงต้นทาง x[n]      : {compute_rmse(x):.5f}")
print(f"RMS ของเสียงไมโครโฟนที่มีเสียงก้อง d[n] : {compute_rmse(d):.5f}")
print(f"RMS ของเสียงคนพูดฝั่งใกล้ target[n]   : {compute_rmse(target):.5f} (ต้องเป็น 0 ในการทดลอง S1)")

In [ ]:
# =============================================================================
# CELL 3 [Student B]: เปรียบเทียบ LMS กับ NLMS ภายใต้พารามิเตอร์เดียวกันเป๊ะ (Same Settings)
# =============================================================================
# สิ่งที่ควรเกิดขึ้นเมื่อรัน Cell นี้:
#   - รันทั้ง Standard LMS และ NLMS ด้วยความยาวตัวกรอง FILTER_TAPS = 128 และ SAME_SETTING_MU = 0.010 เท่ากัน
#   - จะเห็นว่าเมื่อใช้ก้าวการปรับขนาดเล็กที่ปลอดภัยเท่ากัน ทั้ง LMS และ NLMS ลดเสียงก้องได้ใกล้เคียงกัน
# ค่าที่นักศึกษาสามารถลองเปลี่ยนได้อย่างปลอดภัย:
#   - ลองเปลี่ยน SAME_SETTING_MU ในช่วง 0.002 ถึง 0.020 เพื่อดูผลต่อความเร็วการลู่เข้า

FILTER_TAPS = 128       # <--- [Student B ลองเปลี่ยนเป็น 64 หรือ 128 ได้]
SAME_SETTING_MU = 0.010 # <--- [Student B ลองเปลี่ยนในช่วง 0.002 ถึง 0.020 ได้]

p_x, max_win_e, mu_avg_bound, mu_peak_bound = compute_lms_stability_bound(x, FILTER_TAPS)
y_lms, e_lms, w_lms = run_lms(x, d, num_taps=FILTER_TAPS, mu=SAME_SETTING_MU)
y_nlms_same, e_nlms_same, w_nlms_same = run_nlms(x, d, num_taps=FILTER_TAPS, mu=SAME_SETTING_MU)

print(f"ขอบเขตความเสถียรแบบจุดต่อจุดของ LMS บนคลิปนี้ (1 / max ||x[n]||^2) = {mu_peak_bound:.4f}")
print(f"[Standard LMS (N={FILTER_TAPS}, mu={SAME_SETTING_MU})] -> ERLE = {compute_erle_db(d, e_lms):.2f} dB, RMSE = {compute_rmse(e_lms, target):.5f}")
print(f"[NLMS         (N={FILTER_TAPS}, mu={SAME_SETTING_MU})] -> ERLE = {compute_erle_db(d, e_nlms_same):.2f} dB, RMSE = {compute_rmse(e_nlms_same, target):.5f}")

In [ ]:
# =============================================================================
# CELL 4 [Student B]: ทดสอบข้อได้เปรียบของการหารด้วยพลังงานใน NLMS (Normalized Step Size)
# =============================================================================
# สิ่งที่ควรเกิดขึ้นเมื่อรัน Cell นี้:
#   - รัน NLMS ด้วยก้าวปรับตัวขนาดใหญ่ขึ้น (NLMS_NORMALIZED_MU = 0.50) ซึ่งปลอดภัยเพราะมีการหารด้วย ||x[n]||^2 + eps
#   - ค่า ERLE จะพุ่งสูงขึ้นชัดเจน และค่า RMSE ของสัญญาณคงเหลือจะลดลงเข้าใกล้ 0 (ความเงียบสนิท)
# ค่าที่นักศึกษาสามารถลองเปลี่ยนได้อย่างปลอดภัย:
#   - ลองเปลี่ยน NLMS_NORMALIZED_MU ในช่วง 0.10 ถึง 0.80

NLMS_NORMALIZED_MU = 0.50 # <--- [Student B ลองเปลี่ยนในช่วง 0.10 ถึง 0.80 ได้]

y_nlms_norm, e_nlms_norm, w_nlms_norm = run_nlms(x, d, num_taps=FILTER_TAPS, mu=NLMS_NORMALIZED_MU)
print(f"[NLMS (N={FILTER_TAPS}, mu={NLMS_NORMALIZED_MU})] -> ERLE = {compute_erle_db(d, e_nlms_norm):.2f} dB, RMSE = {compute_rmse(e_nlms_norm, target):.5f}")

In [ ]:
# =============================================================================
# CELL 5 [Student A & B]: รันครบทั้ง 10 คลิปและสรุปค่าเฉลี่ย ERLE / RMSE
# =============================================================================
lms_erles, lms_rmses = [], []
nlms_same_erles, nlms_same_rmses = [], []
nlms_norm_erles, nlms_norm_rmses = [], []

for idx, cmeta in enumerate(clips_meta, start=1):
    trial_id = f"S1_T{idx:02d}"
    with wave.open(str(PROJECT_ROOT / str(cmeta["local_path"])), "rb") as wf:
        x_i = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float64) / 32768.0
    d_i, _, tgt_i = simulate_microphone_signal(x_i, [h_room])
    _, e_lms_i, _ = run_lms(x_i, d_i, num_taps=FILTER_TAPS, mu=SAME_SETTING_MU)
    _, e_nlms_same_i, _ = run_nlms(x_i, d_i, num_taps=FILTER_TAPS, mu=SAME_SETTING_MU)
    _, e_nlms_norm_i, _ = run_nlms(x_i, d_i, num_taps=FILTER_TAPS, mu=NLMS_NORMALIZED_MU)

    lms_erles.append(compute_erle_db(d_i, e_lms_i))
    lms_rmses.append(compute_rmse(e_lms_i, tgt_i))
    nlms_same_erles.append(compute_erle_db(d_i, e_nlms_same_i))
    nlms_same_rmses.append(compute_rmse(e_nlms_same_i, tgt_i))
    nlms_norm_erles.append(compute_erle_db(d_i, e_nlms_norm_i))
    nlms_norm_rmses.append(compute_rmse(e_nlms_norm_i, tgt_i))
    print(f"{trial_id}: LMS(mu={SAME_SETTING_MU}) ERLE={lms_erles[-1]:.2f} dB | NLMS(mu={SAME_SETTING_MU}) ERLE={nlms_same_erles[-1]:.2f} dB | NLMS(mu={NLMS_NORMALIZED_MU}) ERLE={nlms_norm_erles[-1]:.2f} dB")

print("-" * 80)
print(f"MEAN +/- SD across 10 clips:")
print(f"  * LMS  (mu={SAME_SETTING_MU:.3f}) : ERLE = {np.mean(lms_erles):.2f} +/- {np.std(lms_erles):.2f} dB | RMSE = {np.mean(lms_rmses):.5f} +/- {np.std(lms_rmses):.5f}")
print(f"  * NLMS (mu={SAME_SETTING_MU:.3f}) : ERLE = {np.mean(nlms_same_erles):.2f} +/- {np.std(nlms_same_erles):.2f} dB | RMSE = {np.mean(nlms_same_rmses):.5f} +/- {np.std(nlms_same_rmses):.5f}")
print(f"  * NLMS (mu={NLMS_NORMALIZED_MU:.2f}) : ERLE = {np.mean(nlms_norm_erles):.2f} +/- {np.std(nlms_norm_erles):.2f} dB | RMSE = {np.mean(nlms_norm_rmses):.5f} +/- {np.std(nlms_norm_rmses):.5f}")

### คำถามซ้อมสอบปากเปล่า (Short Oral-Exam Questions — ฝึกตอบด้วยตนเองโดยไม่มีเฉลยกำกับ)

1. **Causal Buffer Order**: ในโค้ดของตัวกรองขนาด $N$ จุด เวกเตอร์อินพุต `x_vec[n]` เรียงลำดับข้อมูลจากอดีตถึงปัจจุบันอย่างไร และทำไมในช่วงเริ่มต้นที่ $n < N - 1$ เราจึงต้องเติมค่าศูนย์ (Zero-Padding) แทนการวนดึงข้อมูลจากท้ายอาเรย์?
2. **Meaning of ERLE**: ค่า ERLE (dB) คำนวณจากอัตราส่วนของสัญญาณใดเทียบกับสัญญาณใด และถ้าผลการทดลองได้ค่า ERLE เพิ่มขึ้นจาก `14 dB` เป็น `58 dB` มีความหมายต่อเสียงที่เราได้ยินอย่างไร?
3. **Why S1 Residual is Silence**: ทำไมในการทดลอง S1 เมื่อตัวกรอง LMS/NLMS ทำงานได้สมบูรณ์แบบแล้ว สัญญาณคงเหลือ `e[n]` จึงต้องเข้าใกล้ความเงียบสนิท (ค่าใกล้ศูนย์) แทนที่จะได้ยินเสียงพูดชัดเจน?
4. **Why an RIR Change Makes Adaptation Harder**: ถ้าในระหว่างที่ตัวกรองปรับค่าสัมประสิทธิ์ $\mathbf{w}[n]$ จนลู่เข้าหาค่าห้อง `h[n]` เดิมแล้ว จู่ๆ คนพูดหรือไมโครโฟนขยับตำแหน่งทำให้ผลตอบสนองของห้อง `h[n]` เปลี่ยนไปทันที จะเกิดอะไรขึ้นกับสัญญาณคงเหลือ `e[n]` และทำไมการปรับตัวของตัวกรองจึงยากขึ้นกว่ากรณีห้องอยู่นิ่ง?